
# Parallel Segmented Sum on Ascend

> The purpose of this notebook is to test our **new** multi-core segmented sum algorithm for sparse CSR matrices.



In [1]:
import numpy as np
import scipy

from scipy.sparse import random as sp_random
from scipy.sparse import csr_matrix

In [2]:
from tqdm.notebook import tqdm

# Test the result

# Multi-core Segmented Sum Algorithm

> `ref_seg_sum`: a reference segmented sum implementation

> `seg_sum`: is the multi-core segmented sum that we will implement in NPUs.

In [3]:
import numpy as np

def ref_seg_sum(v, indptr):
    "Returns the segment sum of x given segment array `indptr`. The segment array follows the scipy.sparse.csr_matrix conventions."
    num_segments = len(indptr) - 1
    sums = np.zeros(num_segments, dtype=np.float32)

    # For each segment, perform a scan within the segment
    for i in range(1, num_segments + 1):
        s = indptr[i - 1]
        e = indptr[i]
        sums[i - 1] = np.sum(v[s:e])

    return sums

In [4]:
import numpy as np
from math import ceil

def seg_sum(x, indptr, num_blocks: int):
  """Returns the segment sum of x given the indptr segments array from CSR sparse matrix.

  The algorithm works by splitting the problem into multiple blocks as follows:
  1. First block, solve the segmented sum problem in the first block by appending a tail segment as the last segment
  2. On all other blocks, write back the results by overwriting the last entry of the previous block.
  """

  n = len(x)
  num_segments = len(indptr) - 1
  sums = np.zeros(num_segments, dtype=np.float32)

  block_len = ceil(n / num_blocks)
  assert block_len > 1, f"Block length must be greater than 1. Got {block_len} with n = {n}"

  # Block starts are [0, block_len, 2 * block_len, ...]
  seq_start = block_len * np.arange(0, num_blocks + 1)
  # Last se
  seq_start[-1] = min(seq_start[-1], n)

  # Index i of first_indices contains the index on which the i-th block
  # should start consuming from.
  #
  # first_indices has length num_blocks + 1
  block_starts = np.searchsorted(indptr, seq_start)

  for i in range(num_blocks):
    # Range (s,e) is the start and end of block
    s ,e = i * block_len, min((i + 1) * block_len, n)

    # Begin/end indices of indptr for i-th block
    idx_s, idx_e = block_starts[i], block_starts[i + 1]

    # Each block will segsum the block x[s:e] with segments indptr[idx_s:idx_e],
    # hence the indptr values are shifted by 's'
    block_indptr = indptr[idx_s:idx_e] - s

    # Always append an additional segment at the end.
    block_indptr = np.append(block_indptr, len(x[s:e]))

    # Special handling of the first block
    if i > 0:
      # First block iteration writes back output aligned.
      # On all other iterations, overwrite the last entry of
      # the previous block iteration with the first entry of the current block.
      idx_s -= 1
      # Prepend a zero element. Example ind_ptr = [3, 7] becomes
      # [0, 3, 7]
      block_indptr = np.insert(block_indptr, 0, 0)

    # Write back the results with one element overlap between output tiles
    sums[idx_s:idx_e] += ref_seg_sum(x[s:e], block_indptr)

  return sums

## Unit test of seg_sum algorithm over sparse CSR matrices

In [5]:
for m in tqdm([123, 157, 213, 511, 1024]):
#for m in tqdm(range(13, 512)):
    for n in [63, 123, 157, 213, 511, 1023]:
      for density in [0.01, 0.1, 0.3, 0.5]:
        A = sp_random(m, n, density=density, format="csr", dtype=np.float32)
        for num_blocks in [2, 3, 4, 5, 7]:

          actual = seg_sum(A.data, A.indptr, num_blocks)
          expected = ref_seg_sum(A.data, A.indptr)

          assert expected.shape == actual.shape, "Shape must agree"
          assert np.allclose(expected, actual), f"Failed m, n = {m}, {n}. Diff = {expected - actual}"

print("***********************************")
print("* Success. All unit tests passed! *")
print("***********************************")

  0%|          | 0/5 [00:00<?, ?it/s]

***********************************
* Success. All unit tests passed! *
***********************************


In [9]:
import numpy as np
import ssgetpy
import os
from scipy.io import mmread
from scipy.sparse import csr_matrix

SS_HOME = os.environ.get("SPARSE_SUITE_HOME", "/scratch/TCUSCAN/sparse-suite-matrices/ssgetpy-downloaded-matrices/")


matrices = ssgetpy.search(nzbounds=(100000, 1000000), colbounds=(10000, 70000), limit=900)

for matrix in tqdm(matrices):
    matrix_id = matrix.id
    matrix = ssgetpy.fetch(matrix_id, location=SS_HOME)[0]
    file_location, _ = matrix.download(extract=True, destpath=SS_HOME)
    mat_full_path = os.path.join(file_location, f"{matrix.name}.mtx")
    filename = f"{mat_full_path}.mtx"
    A = csr_matrix(mmread(mat_full_path))
    # Normalize data in [-std, +std]
    A.data = (A.data - np.mean(A.data)) / np.std(A.data)

    if np.any(np.isnan(A.data)) or np.any(np.iscomplex(A.data)):
        print(f"Input sparse matrix {matrix.name} has NaN or complex values.")
        continue


    for num_blocks in [5, 7, 20, 24]:

        actual = seg_sum(A.data, A.indptr, num_blocks)
        expected = ref_seg_sum(A.data, A.indptr)

        assert expected.shape == actual.shape, "Shape must agree"
        # Fails due to precision. Aleks won't be happy about it :-(
        # assert np.allclose(expected, actual), f"Failed m, n = {m}, {n}. Norm = {np.mean(expected - actual)}"
        assert np.mean(np.abs(expected - actual)) < 1e-4, f"Failed m, n = {m}, {n}. Norm = {np.mean(expected - actual)}"

print("***********************************")
print("* Success. All unit tests passed! *")
print("***********************************")

  0%|          | 0/390 [00:00<?, ?it/s]

/tmp/ipykernel_2091096/3511580356.py:20: RuntimeWarning: invalid value encountered in divide
  A.data = (A.data - np.mean(A.data)) / np.std(A.data)


Input sparse matrix bcsstk29 has NaN or complex values.
Input sparse matrix pcrystk02 has NaN or complex values.
Input sparse matrix pwt has NaN or complex values.
Input sparse matrix shuttle_eddy has NaN or complex values.
Input sparse matrix skirt has NaN or complex values.
Input sparse matrix pkustk01 has NaN or complex values.
Input sparse matrix pkustk02 has NaN or complex values.
Input sparse matrix kim1 has NaN or complex values.
Input sparse matrix barth5 has NaN or complex values.
Input sparse matrix pwt has NaN or complex values.
Input sparse matrix shuttle_eddy has NaN or complex values.
Input sparse matrix skirt has NaN or complex values.
Input sparse matrix tandem_vtx has NaN or complex values.
Input sparse matrix waveguide3D has NaN or complex values.
Input sparse matrix rajat09 has NaN or complex values.
Input sparse matrix rajat10 has NaN or complex values.
Input sparse matrix copter1 has NaN or complex values.
Input sparse matrix crplat2 has NaN or complex values.
Inpu